# R2：PyTorch 张量与自动求导

本节目标：

1. 知道 Tensor 是 PyTorch 用来保存数据和参数的对象。
2. 理解 `requires_grad=True` 表示要追踪这个参数参与的计算。
3. 理解 `loss.backward()` 会按链式法则计算梯度，并把结果放入参数的 `.grad`。

不要求记住所有 API；重点是把你已学过的“损失 → 梯度 → 更新参数”训练循环，和 PyTorch 的写法对应起来。

## 1. Tensor：带计算能力的数据容器

Tensor 与 NumPy array 都能表示向量、矩阵等数据。PyTorch Tensor 额外支持：在 GPU 上计算，以及自动记录计算过程来求梯度。

In [1]:
import torch

# 一个普通 Tensor：保存数据，默认不追踪梯度。
features = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print('features =')
print(features)
print('shape:', features.shape)
print('requires_grad:', features.requires_grad)

features =
tensor([[1., 2.],
        [3., 4.]])
shape: torch.Size([2, 2])
requires_grad: False


## 2. 让 PyTorch 自动求一个已知导数

设预测公式为：

$$
\hat{y}=3w
$$

真实目标是 6，损失为：

$$
L=(3w-6)^2
$$

你以前手推过：

$$
\frac{dL}{dw}=2(3w-6)\cdot3
$$

当 `w=1` 时，梯度为 -18。下面让 PyTorch 算同一件事。

In [2]:
# w 是待学习参数，因此 requires_grad=True。
w = torch.tensor(1.0, requires_grad=True)
target = torch.tensor(6.0)

prediction = 3 * w
loss = (prediction - target) ** 2

print('prediction:', prediction.item())
print('loss:', loss.item())
print('backward 前，w.grad:', w.grad)

prediction: 3.0
loss: 9.0
backward 前，w.grad: None


In [3]:
# backward() 沿着 prediction 和 loss 的计算过程反向计算 dL/dw。
loss.backward()

# 梯度保存在叶子参数 w 的 .grad 属性中。
print('PyTorch 计算的 dL/dw:', w.grad.item())
print('手推答案应为: -18')

PyTorch 计算的 dL/dw: -18.0
手推答案应为: -18


## 3. 使用梯度更新一次参数

梯度下降仍然是同一个公式：

$$
w\leftarrow w-\eta\frac{dL}{dw}
$$

这里先手动更新一次，方便看到 PyTorch 自动求导与之前的数学知识如何连接。后续会使用 `torch.optim` 自动完成更新。

In [4]:
learning_rate = 0.1

# 更新参数本身不是模型计算图的一部分，因此放到 no_grad() 中。
with torch.no_grad():
    w -= learning_rate * w.grad

# PyTorch 默认累积梯度；每轮训练后必须清零，防止和下一轮梯度相加。
w.grad.zero_()

new_prediction = 3 * w
new_loss = (new_prediction - target) ** 2
print('更新后的 w:', w.item())
print('更新后的 prediction:', new_prediction.item())
print('更新后的 loss:', new_loss.item())

更新后的 w: 2.8000001907348633
更新后的 prediction: 8.40000057220459
更新后的 loss: 5.760002613067627


## 本节训练循环映射

```python
prediction = model(x)   # 前向计算
loss = loss_function(prediction, y)  # 计算损失
loss.backward()         # 反向传播：计算梯度
optimizer.step()        # 用梯度更新参数
optimizer.zero_grad()   # 清空已累积的梯度
```

下节会把这五步放进一个完整的 PyTorch 线性回归训练循环。